In [6]:
!rm -rf /content/models

In [1]:
!pip uninstall -y torchvision
!pip install torchvision==0.20.1+cu130 --index-url https://download.pytorch.org/whl/cu130
!pip install -q evaluate

Looking in indexes: https://download.pytorch.org/whl/cu130
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1+cu130 (from versions: 0.1.6, 0.2.0, 0.24.0+cu130, 0.24.1+cu130, 0.25.0+cu130, 0.26.0+cu130)
ERROR: No matching distribution found for torchvision==0.20.1+cu130


In [2]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -U transformers datasets seqeval torch tqdm

# =========================================
# 2️⃣ Imports
# =========================================
import json
import re
import torch
import numpy as np
from tqdm import tqdm
import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
from sklearn.model_selection import train_test_split

seqeval_metric = evaluate.load("seqeval")

# =========================================
# 3️⃣ Load Dataset
# =========================================
dataset_path = "/content/CN-cleaned_file.jsonl"

dataset = []
with open(dataset_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [d["input"] for d in dataset]
labels = [json.loads(d["output"]) for d in dataset]

# =========================================
# 4️⃣ BIO Label Setup
# =========================================
entity_types = ["TOPIC", "CONCEPT", "METHOD", "ALGORITHM"]

label_list = ["O"]
for t in entity_types:
    label_list.append(f"B-{t}")
    label_list.append(f"I-{t}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

# =========================================
# 5️⃣ Text Normalization
# =========================================
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", "", text)
    return text

# =========================================
# 6️⃣ Encode Labels (BIO FIXED)
# =========================================
def encode_labels(text, entities):
    tokens = text.split()
    token_labels = ["O"] * len(tokens)

    for ent in entities:
        ent_text = normalize(ent["entity"])
        ent_tokens = ent_text.split()
        ent_label = ent["label"] if ent["label"] in entity_types else "CONCEPT"

        for i in range(len(tokens) - len(ent_tokens) + 1):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                token_labels[i] = f"B-{ent_label}"
                for j in range(1, len(ent_tokens)):
                    token_labels[i+j] = f"I-{ent_label}"

    return token_labels

# Normalize texts
texts = [normalize(t) for t in texts]

token_labels = [encode_labels(t, e) for t, e in zip(texts, labels)]

# =========================================
# 7️⃣ Split Dataset
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, token_labels, test_size=0.15, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.33, random_state=42
)

train_dataset = Dataset.from_dict({"tokens": [t.split() for t in train_texts], "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": [t.split() for t in val_texts], "ner_tags": val_labels})
test_dataset = Dataset.from_dict({"tokens": [t.split() for t in test_texts], "ner_tags": test_labels})

# =========================================
# 8️⃣ Models
# =========================================
scibert_name = "allenai/scibert_scivocab_uncased"
roberta_name = "roberta-base"

scibert_tokenizer = AutoTokenizer.from_pretrained(scibert_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

scibert_model = AutoModelForTokenClassification.from_pretrained(scibert_name, num_labels=len(label_list))
roberta_model = AutoModelForTokenClassification.from_pretrained(roberta_name, num_labels=len(label_list))

for model in [scibert_model, roberta_model]:
    model.config.id2label = id2label
    model.config.label2id = label2id

# =========================================
# 9️⃣ Tokenization
# =========================================
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_id]])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_sci = train_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)
val_sci = val_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)

train_rob = train_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)
val_rob = val_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)

for d in [train_sci, val_sci, train_rob, val_rob]:
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# =========================================
# 🔟 Training
# =========================================
args = TrainingArguments(
    output_dir="./models",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",   # ✅ FIXED
    save_strategy="epoch",
    logging_dir="./logs"
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true = [[id2label[l] for l in lab if l != -100] for lab in labels]
    pred = [[id2label[p] for p, l in zip(pr, lab) if l != -100] for pr, lab in zip(preds, labels)]

    return seqeval_metric.compute(predictions=pred, references=true)

trainer_sci = Trainer(model=scibert_model, args=args, train_dataset=train_sci, eval_dataset=val_sci, compute_metrics=compute_metrics)
trainer_rob = Trainer(model=roberta_model, args=args, train_dataset=train_rob, eval_dataset=val_rob, compute_metrics=compute_metrics)

trainer_sci.train()
trainer_rob.train()

# =========================================
# 1️⃣1️⃣ Pipelines
# =========================================
sci_pipe = pipeline("ner", model=scibert_model, tokenizer=scibert_tokenizer, aggregation_strategy="simple")
rob_pipe = pipeline("ner", model=roberta_model, tokenizer=roberta_tokenizer, aggregation_strategy="simple")

# =========================================
# 1️⃣2️⃣ Strong Filtering
# =========================================
STOPWORDS = {"the", "and", "or", "to", "of", "in", "a"}

def clean_entity(text):
    text = text.lower().strip()
    if text in STOPWORDS:
        return None
    if len(text) < 3:
        return None
    if text.endswith(("and", "or")):
        return None
    if "," in text:
        return None
    return text

# =========================================
# 1️⃣3️⃣ Ensemble (IMPROVED)
# =========================================
def ensemble(sci, rob):
    combined = {}

    for e in sci:
        key = clean_entity(e["word"])
        if not key: continue
        combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "sci"}

    for e in rob:
        key = clean_entity(e["word"])
        if not key: continue

        if key in combined:
            combined[key]["score"] = max(combined[key]["score"], e["score"])
            combined[key]["source"] = "both"
        else:
            if e["score"] > 0.75:
                combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "rob"}

    # keep strong ones
    final = []
    for k, v in combined.items():
        if v["source"] == "both" or v["score"] > 0.8:
            final.append({
                "entity": k,
                "label": v["label"],
                "score": round(float(v["score"]), 3)
            })

    return final

# =========================================
# 1️⃣4️⃣ Run Ensemble
# =========================================
results = []

for text in tqdm(test_texts):
    sci = sci_pipe(text)
    rob = rob_pipe(text)

    ents = ensemble(sci, rob)

    results.append({
        "text": text,
        "entities": ents
    })

# =========================================
# 1️⃣5️⃣ Save
# =========================================
with open("FINAL_KG_READY.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ CLEAN KG FILE READY")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored 

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 76/76 [00:02<00:00, 31.89it/s]

✅ CLEAN KG FILE READY


In [ ]:
# =========================================
# 📊 ENSEMBLE EVALUATION CELL
# =========================================

import json
import numpy as np
import evaluate
from tqdm import tqdm

seqeval_metric = evaluate.load("seqeval")

# =========================================
# 1️⃣ Load predictions
# =========================================
with open("FINAL_KG_READY.json", "r") as f:
    ensemble_results = json.load(f)

# =========================================
# 2️⃣ Helper: rebuild BIO from ensemble
# =========================================
def align_to_bio(text, entities):
    tokens = text.split()
    labels = ["O"] * len(tokens)

    for ent in entities:
        ent_tokens = ent["entity"].split()
        ent_label = ent["label"]

        for i in range(len(tokens) - len(ent_tokens) + 1):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                labels[i] = f"B-{ent_label}"
                for j in range(1, len(ent_tokens)):
                    labels[i+j] = f"I-{ent_label}"

    return labels

# =========================================
# 3️⃣ Build predictions + references
# =========================================
y_true = []
y_pred = []

for i, item in enumerate(tqdm(ensemble_results)):
    text = item["text"]
    pred_entities = item["entities"]

    # predicted BIO
    pred_bio = align_to_bio(text, pred_entities)

    # TRUE labels from test_labels
    true_bio = test_labels[i]

    y_true.append(true_bio)
    y_pred.append(pred_bio)

# =========================================
# 4️⃣ Fix label alignment (safety check)
# =========================================
def clean_labels(batch):
    return [
        [l if l in label_list else "O" for l in seq]
        for seq in batch
    ]

y_true = clean_labels(y_true)
y_pred = clean_labels(y_pred)

# =========================================
# 5️⃣ Metrics
# =========================================
results = seqeval_metric.compute(
    predictions=y_pred,
    references=y_true
)

print("\n📊 ===== ENSEMBLE RESULTS =====")
print(f"Precision: {results['overall_precision']:.4f}")
print(f"Recall   : {results['overall_recall']:.4f}")
print(f"F1-score : {results['overall_f1']:.4f}")
print(f"Accuracy : {results['overall_accuracy']:.4f}")

100%|██████████| 76/76 [00:00<00:00, 10451.04it/s]


📊 ===== ENSEMBLE RESULTS =====
Precision: 0.8847
Recall   : 0.7833
F1-score : 0.8309
Accuracy : 0.9771
